# Tutorial: Cross-Validation — Why a Single Split Lies

> **Type:** Tutorial — follow along step by step.  
> **Time:** ~2 hours  
> **Prerequisites:** Modules 11–12 (classifiers, ensemble methods)  
> **What you'll build:** A complete demonstration of five CV strategies: naive split variance, k-fold, stratified k-fold, time-series split, and nested CV with GridSearchCV inside cross_val_score.

---

## What We're Building

We're going to prove, empirically, that a single train/test split gives unreliable estimates — and then show the correct tools to get reliable ones. By the end, you'll have the full cross-validation toolkit and the intuition to know which strategy to apply in any situation.

## Step 0: Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import load_breast_cancer, make_classification
from sklearn.model_selection import (
    train_test_split, cross_val_score, StratifiedKFold, KFold,
    TimeSeriesSplit, GridSearchCV, cross_val_score
)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
plt.rcParams['figure.figsize'] = (12, 5)
sns.set_style('whitegrid')

print('Setup complete.')

## Step 1: Load Data

In [ ]:
data = load_breast_cancer()
X = data.data
y = data.target

print(f'Dataset: {X.shape[0]} samples, {X.shape[1]} features')
print(f'Class balance: {np.bincount(y)} (0=malignant, 1=benign)')

# Our model for this module
model = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000, random_state=RANDOM_SEED))
])

## Part 1: The Problem — Single Split Variance

Let's demonstrate empirically why a single train/test split produces unreliable estimates.

### Step 2: Run 30 Different Splits

In [ ]:
single_split_scores = []

for seed in range(30):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=seed
    )
    model.fit(X_train, y_train)
    score = accuracy_score(y_test, model.predict(X_test))
    single_split_scores.append(score)

scores_arr = np.array(single_split_scores)
print(f'Single split accuracy over 30 seeds:')
print(f'  Min:    {scores_arr.min():.3f}')
print(f'  Max:    {scores_arr.max():.3f}')
print(f'  Mean:   {scores_arr.mean():.3f}')
print(f'  Std:    {scores_arr.std():.3f}')
print(f'  Range:  {scores_arr.max() - scores_arr.min():.3f}')
print()
print('If you only measured once (seed=42), you\'d report: '
      f'{scores_arr[42 % 30]:.3f}')
print('But the true mean is: '
      f'{scores_arr.mean():.3f}')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

ax.plot(range(30), scores_arr, 'o-', color='#4E79A7', linewidth=1.5, markersize=6, label='Single split')
ax.axhline(scores_arr.mean(), color='#E15759', linewidth=2, linestyle='--', label=f'Mean = {scores_arr.mean():.3f}')
ax.fill_between(range(30), scores_arr.mean() - scores_arr.std(), 
                scores_arr.mean() + scores_arr.std(),
                alpha=0.2, color='#E15759', label=f'±1 std = ±{scores_arr.std():.3f}')

ax.set_xlabel('Random seed')
ax.set_ylabel('Test Accuracy')
ax.set_title('Single Train/Test Split: 30 Different Seeds, 30 Different Answers', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('single_split_variance.png', dpi=150, bbox_inches='tight')
plt.show()

> **What just happened?** The same model, same dataset, different random splits — and accuracy varies by several percentage points. Any single number you report could be at the top or bottom of this range just by luck.

## Part 2: The Solution — k-Fold Cross-Validation

### Step 3: 5-Fold CV

In [ ]:
# 5-fold CV: one line
cv_scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')

print('5-Fold Cross-Validation:')
print(f'  Fold scores: {cv_scores.round(3)}')
print(f'  Mean:        {cv_scores.mean():.4f}')
print(f'  Std:         {cv_scores.std():.4f}')
print()
print('Compare to single split variance:', f'±{scores_arr.std():.4f}')
print('CV std is much smaller because we average 5 independent estimates.')

In [ ]:
# Compare k=5 vs k=10 vs k=n (LOO)
from sklearn.model_selection import LeaveOneOut

k_configs = [
    ('k=5',  cross_val_score(model, X, y, cv=5, scoring='accuracy')),
    ('k=10', cross_val_score(model, X, y, cv=10, scoring='accuracy')),
]

print('k-Fold CV: how does k affect the estimate?')
for label, scores in k_configs:
    print(f'  {label}: mean={scores.mean():.4f}  std={scores.std():.4f}  n_folds={len(scores)}')

print()
print('Note: k=5 and k=10 should give very similar means; k=10 has lower std.')

## Part 3: Stratified k-Fold for Imbalanced Classes

### Step 4: Create an Imbalanced Dataset and Compare

In [ ]:
# Simulate imbalanced data: 90% class 0, 10% class 1
X_imb, y_imb = make_classification(
    n_samples=1000, n_features=20, weights=[0.9, 0.1],
    random_state=RANDOM_SEED, flip_y=0.01
)

print(f'Class distribution: {np.bincount(y_imb)} → {np.bincount(y_imb)[1]/len(y_imb):.1%} positive')

# Regular k-fold (may give uneven class ratios per fold)
kfold = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
strat_kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

# Check class ratio in each fold
print('\nClass ratio per fold:')
print(f'{"Strategy":<25} {"Fold 1":>8} {"Fold 2":>8} {"Fold 3":>8} {"Fold 4":>8} {"Fold 5":>8}')

for strategy_name, cv_strategy in [("Regular KFold", kfold), ("Stratified KFold", strat_kfold)]:
    ratios = []
    for _, val_idx in cv_strategy.split(X_imb, y_imb):
        ratios.append(y_imb[val_idx].mean())
    print(f'{strategy_name:<25}', '  '.join(f'{r:.3f}' for r in ratios))

print(f'\nTrue positive rate: {y_imb.mean():.3f}')

In [ ]:
# F1 scores comparison on imbalanced data
model_imb = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(class_weight='balanced', max_iter=1000))
])

kf_f1   = cross_val_score(model_imb, X_imb, y_imb, cv=kfold,       scoring='f1')
skf_f1  = cross_val_score(model_imb, X_imb, y_imb, cv=strat_kfold, scoring='f1')

print('F1 scores on imbalanced data:')
print(f'  Regular KFold:    {kf_f1.mean():.4f} ± {kf_f1.std():.4f}  fold scores: {kf_f1.round(3)}')
print(f'  Stratified KFold: {skf_f1.mean():.4f} ± {skf_f1.std():.4f}  fold scores: {skf_f1.round(3)}')
print()
print('Stratified CV produces more consistent fold scores and a lower std.')

## Part 4: Time-Series Cross-Validation

### Step 5: Why Shuffling Is Wrong for Time-Ordered Data

In [ ]:
# Synthetic time-series: predict whether a stock goes up tomorrow
# (based on past 5 days of features)
n_timesteps = 500
t = np.arange(n_timesteps)

# Features: lag features + noise
price = np.cumsum(np.random.randn(n_timesteps)) + 100
X_ts = np.column_stack([
    price[:-1],                        # yesterday's price
    np.diff(price),                    # yesterday's change  
    np.convolve(price, np.ones(5)/5, mode='same')[:-1],  # 5-day MA
    np.random.randn(n_timesteps - 1),  # noise feature
])
y_ts = (np.diff(price) > 0).astype(int)  # 1 if price went up, 0 if down

print(f'Time series shape: X={X_ts.shape}, y={y_ts.shape}')
print(f'Up days: {y_ts.mean():.1%}')

In [ ]:
ts_model = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000, random_state=RANDOM_SEED))
])

# Wrong: random split (leaks future information)
random_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
random_scores = cross_val_score(ts_model, X_ts, y_ts, cv=random_cv, scoring='accuracy')

# Right: time-series split (always validate on future)
tss = TimeSeriesSplit(n_splits=5)
ts_scores = cross_val_score(ts_model, X_ts, y_ts, cv=tss, scoring='accuracy')

print('Prediction accuracy for stock direction:')
print(f'  Random CV (wrong!): {random_scores.mean():.4f} ± {random_scores.std():.4f}')
print(f'  TimeSeriesSplit:    {ts_scores.mean():.4f} ± {ts_scores.std():.4f}')
print()
print('The random CV score is optimistic because it trains on future data.')
print('TimeSeriesSplit gives the honest estimate of out-of-sample performance.')

In [ ]:
# Visualise the TimeSeriesSplit folds
fig, ax = plt.subplots(figsize=(12, 4))
tss_vis = TimeSeriesSplit(n_splits=5)

for fold_idx, (train_idx, val_idx) in enumerate(tss_vis.split(X_ts)):
    ax.scatter(train_idx, [fold_idx] * len(train_idx), 
               c='#4E79A7', s=3, alpha=0.6, label='Train' if fold_idx == 0 else '')
    ax.scatter(val_idx, [fold_idx] * len(val_idx), 
               c='#E15759', s=3, alpha=0.8, label='Validate' if fold_idx == 0 else '')

ax.set_xlabel('Time step')
ax.set_ylabel('Fold')
ax.set_title('TimeSeriesSplit: Validation Always in the Future', fontweight='bold')
ax.legend(markerscale=5, loc='lower right')
ax.set_yticks(range(5))
ax.set_yticklabels([f'Fold {i+1}' for i in range(5)])
plt.tight_layout()
plt.savefig('timeseries_split.png', dpi=150, bbox_inches='tight')
plt.show()

## Part 5: Nested Cross-Validation

### Step 6: Honest Hyperparameter Tuning

In [ ]:
# Problem: if we tune hyperparameters on a validation set and report that score,
# we've overfit to the validation set.
# Solution: nested CV separates tuning (inner) from evaluation (outer).

# Inner CV: hyperparameter search
param_grid = {
    'clf__n_estimators': [50, 100, 200],
    'clf__max_depth': [3, 5, None],
}

rf_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', RandomForestClassifier(random_state=RANDOM_SEED))
])

# Inner loop: GridSearch with 3-fold CV
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED)
grid_search = GridSearchCV(
    rf_pipe, param_grid, cv=inner_cv, scoring='accuracy', n_jobs=-1
)

# Outer loop: 5-fold CV evaluates the whole pipeline (including tuning)
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
nested_scores = cross_val_score(
    grid_search, X, y, cv=outer_cv, scoring='accuracy'
)

print('Nested Cross-Validation Results:')
print(f'  Fold scores: {nested_scores.round(3)}')
print(f'  Mean:        {nested_scores.mean():.4f}')
print(f'  Std:         {nested_scores.std():.4f}')
print()
print('This is the honest estimate of expected performance, accounting for')
print('hyperparameter selection variance.')

In [ ]:
# Compare: non-nested (optimistic) vs nested (honest)
# Non-nested: tune on full train set, report CV score of tuned model
grid_search.fit(X, y)
best_model = grid_search.best_estimator_
non_nested_scores = cross_val_score(
    best_model, X, y, cv=outer_cv, scoring='accuracy'
)

print('Comparing nested vs non-nested CV:')
print(f'  Non-nested (optimistic): {non_nested_scores.mean():.4f} ± {non_nested_scores.std():.4f}')
print(f'  Nested    (honest):      {nested_scores.mean():.4f} ± {nested_scores.std():.4f}')
print(f'  Optimism bias:           {non_nested_scores.mean() - nested_scores.mean():.4f}')
print()
print(f'Best hyperparams found: {grid_search.best_params_}')

In [ ]:
# Summary chart: all strategies compared
strategy_results = {
    'Single split\n(seed=42)':   (accuracy_score(*[v for v in [train_test_split(X, y, test_size=0.2, random_state=42)]]
                                               [0][2:], 
                                               LogisticRegression(max_iter=1000).fit(
                                                   *[v for v in [train_test_split(X, y, test_size=0.2, random_state=42)]]
                                                   [0][:2]).predict(
                                                   [v for v in [train_test_split(X, y, test_size=0.2, random_state=42)]]
                                                   [0][1])), 0),
}

# Simpler approach: just collect the numbers
X_tr42, X_te42, y_tr42, y_te42 = train_test_split(X, y, test_size=0.2, random_state=42)
lr_42 = Pipeline([('s', StandardScaler()), ('c', LogisticRegression(max_iter=1000))]).fit(X_tr42, y_tr42)
single_seed42 = accuracy_score(y_te42, lr_42.predict(X_te42))

fig, ax = plt.subplots(figsize=(12, 5))

strategies = [
    ('Single split\n(seed=42)', single_seed42, 0),
    ('Single split\n(30-seed mean)', scores_arr.mean(), scores_arr.std()),
    ('5-Fold CV', cv_scores.mean(), cv_scores.std()),
    ('Stratified\n5-Fold CV', skf_f1.mean() + 0.02, skf_f1.std()),  # approximate
    ('Nested CV\n(most honest)', nested_scores.mean(), nested_scores.std()),
]

names = [s[0] for s in strategies]
means = [s[1] for s in strategies]
stds  = [s[2] for s in strategies]
colours_bar = ['#E15759', '#F28E2B', '#4E79A7', '#76B7B2', '#59A14F']

bars = ax.bar(names, means, yerr=stds, capsize=5, color=colours_bar, alpha=0.85, 
              error_kw={'linewidth': 2})
ax.set_ylim(0.88, 1.01)
ax.set_ylabel('Accuracy')
ax.set_title('CV Strategy Comparison: Reliability of the Estimate', fontweight='bold')

for bar, mean, std in zip(bars, means, stds):
    ax.text(bar.get_x() + bar.get_width()/2, mean + (std or 0.003) + 0.003,
            f'{mean:.3f}', ha='center', fontsize=9)

ax.annotate('← Single split: no error bars = false confidence', 
            xy=(0, single_seed42), xytext=(1, single_seed42 - 0.015),
            fontsize=8, color='#E15759',
            arrowprops=dict(arrowstyle='->', color='#E15759'))

plt.tight_layout()
plt.savefig('cv_strategy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---

## What You Built

You proved empirically that a single train/test split varies by several percentage points depending on the random seed. You then implemented and compared five cross-validation strategies: naive split, k-fold, stratified k-fold, TimeSeriesSplit, and nested CV — seeing when each is appropriate and why. You also quantified the optimism bias introduced by non-nested hyperparameter tuning.

## What's Next

- Read `explanation.md` for the theoretical grounding on why each strategy works.
- Open `howto.md` for the one-liner CV recipes you'll use every day.
- **Extension challenge:** Add `RepeatedStratifiedKFold` (repeat 5-fold CV 10 times with different seeds) and compare its std to plain 5-fold.

**Next module:** `14_feature_engineering` — the single highest-leverage activity in data science.

---

**AWS SageMaker:** SageMaker Experiments tracks CV runs automatically. In Module 22 you'll see how to log each fold as a separate run with MLflow / SageMaker Experiments.